In [2]:
import cv2
import numpy as np
import collections
import mediapipe as mp
from ultralytics import YOLO
import matplotlib.pyplot as plt

#  ─────────────────────────────────────────
YOLO_MODEL_PATH       = "yolov8n.pt"
CAMERA_INDEX          = 0
SEQUENCE_LENGTH       = 30         
HISTORY_LEN           = 10         
CURL_HISTORY_LEN      = 5
MIN_DETECT_CONF       = 70          
STRONG_GRIP_CURL      = 75        
WEAK_GRIP_CURL        = 35        
CURL_WEIGHT           = 0.50
STABILITY_WEIGHT      = 0.30
TREMOR_WEIGHT         = 0.20
STABILITY_SCALE       = 2000        
TREMOR_SCALE          = 1000        
GRIP_OBJECTS          = {"pen", "bottle", "phone", "book", "remote"}

# ─────────────────────────────────────────
COLOR_STRONG  = (0, 255, 0)
COLOR_WEAK    = (0, 165, 255)
COLOR_NONE    = (0, 0, 255)
COLOR_OBJECT  = (255, 165, 0)
COLOR_TEXT    = (255, 255, 255)
COLOR_METRIC  = (255, 255, 0)
COLOR_PANEL   = (50, 50, 50)



def load_models():
    yolo = YOLO(YOLO_MODEL_PATH)
    print("✅ YOLO loaded!")

    mp_hands_mod = mp.solutions.hands
    hands = mp_hands_mod.Hands(
        max_num_hands=1,
        min_detection_confidence=0.7,
    )
    print("✅ MediaPipe loaded!")
    return yolo, hands


def detect_object(frame, yolo_model, object_history: collections.deque):
    
    results = yolo_model(frame, verbose=False)

    best_name, best_box, best_conf = None, None, 0.0

    for result in results:
        for box in result.boxes:
            class_id   = int(box.cls[0])
            name       = yolo_model.names[class_id].lower()
            confidence = float(box.conf[0]) * 100

            if any(obj in name for obj in GRIP_OBJECTS) and confidence > best_conf:
                best_name, best_box, best_conf = name, box, confidence

    if best_name and best_conf >= MIN_DETECT_CONF:
        object_history.append(best_name)
        stable = max(set(object_history), key=object_history.count)
        return stable, best_box, best_conf

    return None, None, 0.0


def compute_curl(sequence: np.ndarray) -> float:
   
    FINGER_PAIRS = [(8, 5), (12, 9), (16, 13), (20, 17)]
    curl_per_frame = []

    for frame_row in sequence:
        pts    = frame_row.reshape(21, 3)
        curled = sum(pts[tip][1] > pts[mcp][1] for tip, mcp in FINGER_PAIRS)
        curl_per_frame.append(curled / len(FINGER_PAIRS))

    return float(np.mean(curl_per_frame)) * 100


def compute_stability(sequence: np.ndarray) -> float:
    wrist = sequence[:, :3]
    return float(np.clip(100 - np.std(wrist) * STABILITY_SCALE, 0, 100))


def compute_tremor(sequence: np.ndarray) -> float:
    """Frame-to-frame landmark movement → tremor resistance %."""
    diffs = np.diff(sequence, axis=0)
    return float(np.clip(100 - np.mean(np.abs(diffs)) * TREMOR_SCALE, 0, 100))


def classify_grip(smooth_curl: float) -> str:
    if smooth_curl >= STRONG_GRIP_CURL:
        return "Strong Grip"
    if smooth_curl >= WEAK_GRIP_CURL:
        return "Weak Grip"
    return "No Grip"


def grip_color(grip: str) -> tuple:
    return {"Strong Grip": COLOR_STRONG,
            "Weak Grip":   COLOR_WEAK,
            "No Grip":     COLOR_NONE}.get(grip, COLOR_NONE)


def score_color(score: int) -> tuple:
    if score >= 70:
        return COLOR_STRONG
    if score >= 40:
        return COLOR_WEAK
    return COLOR_NONE



def draw_object_box(frame, label: str, conf: float, box):
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    cv2.rectangle(frame, (x1, y1), (x2, y2), COLOR_OBJECT, 2)
    cv2.putText(frame, f"{label} {conf:.0f}%",
                (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, COLOR_OBJECT, 2)


def draw_ui(frame, detected_object: str, current_grip: str,
            avg_score: int, smooth_curl: float,
            stability: float, tremor: float):
    """Semi-transparent panel + all metric labels."""
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (450, 330), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

    g_col = grip_color(current_grip)
    s_col = score_color(avg_score)

    labels = [
        ("GripSense",                    35,  0.9, COLOR_TEXT,   2),
        (f"Object : {detected_object}",  75,  0.8, COLOR_OBJECT, 2),
        (f"Grip   : {current_grip}",     110, 0.8, g_col,        2),
        (f"Score  : {avg_score}/100",    148, 0.9, s_col,        2),
    ]
    for text, y, scale, color, thick in labels:
        cv2.putText(frame, text, (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick)

    # Score bar
    bar_w = int((avg_score / 100) * 380)
    cv2.rectangle(frame, (10, 165), (390, 190), COLOR_PANEL, -1)
    cv2.rectangle(frame, (10, 165), (10 + bar_w, 190), s_col, -1)

    # Detail metrics
    for text, y in [
        (f"Curl     : {int(smooth_curl)}%", 225),
        (f"Stability: {int(stability)}%",   250),
        (f"Tremor   : {int(tremor)}%",      275),
    ]:
        cv2.putText(frame, text, (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, COLOR_METRIC, 1)

    cv2.putText(frame, "Press Q to quit",
                (10, frame.shape[0] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)








def main():
    yolo_model, hands = load_models()
    mp_draw = mp.solutions.drawing_utils

    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open camera index {CAMERA_INDEX}")
    print("GripSense Running... Press Q to quit")

    # Rolling buffers — all bounded, no manual .pop(0)
    sequence_buffer = collections.deque(maxlen=SEQUENCE_LENGTH)
    score_history   = collections.deque(maxlen=HISTORY_LEN)
    object_history  = collections.deque(maxlen=HISTORY_LEN)
    grip_history    = collections.deque(maxlen=HISTORY_LEN)
    curl_history    = collections.deque(maxlen=CURL_HISTORY_LEN)

    # Persistent display state
    detected_object = "No Object"
    current_grip    = "Detecting..."
    avg_score       = 0
    smooth_curl     = 0.0
    stability       = 0.0
    tremor          = 0.0


    session_scores    = []
    session_curls     = []
    session_stability = []
    session_tremor    = []
    session_frames    = []
    frame_count       = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Camera read failed — exiting.")
            break

        
        obj_name, obj_box, obj_conf = detect_object(frame, yolo_model, object_history)
        if obj_name:
            detected_object = obj_name
            draw_object_box(frame, detected_object, obj_conf, obj_box)

        
        rgb         = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        hand_result = hands.process(rgb)

        if hand_result.multi_hand_landmarks:
            for lm_set in hand_result.multi_hand_landmarks:
                mp_draw.draw_landmarks(
                    frame, lm_set, mp.solutions.hands.HAND_CONNECTIONS
                )

                row = [c for lm in lm_set.landmark for c in (lm.x, lm.y, lm.z)]
                sequence_buffer.append(row)

                if len(sequence_buffer) == SEQUENCE_LENGTH:
                    pts = np.array(sequence_buffer)

                    curl       = compute_curl(pts)
                    stability  = compute_stability(pts)
                    tremor     = compute_tremor(pts)

                    curl_history.append(curl)
                    smooth_curl = float(np.mean(curl_history))

                    score = int(np.clip(
                        smooth_curl * CURL_WEIGHT
                        + stability * STABILITY_WEIGHT
                        + tremor    * TREMOR_WEIGHT,
                        0, 100,
                    ))
                    score_history.append(score)
                    avg_score = int(np.mean(score_history))

                    grip_history.append(classify_grip(smooth_curl))
                    current_grip = max(set(grip_history), key=grip_history.count)

                    # ── Collect real data every frame ──────────────────
                    session_scores.append(avg_score)
                    session_curls.append(int(smooth_curl))
                    session_stability.append(int(stability))
                    session_tremor.append(int(tremor))
                    session_frames.append(frame_count)

                    draw_ui(frame, detected_object, current_grip,
                            avg_score, smooth_curl, stability, tremor)

        else:
            sequence_buffer.clear()
            grip_history.clear()
            cv2.putText(frame, "No Hand Detected!",
                        (10, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        0.9, COLOR_NONE, 2)

        frame_count += 1

        cv2.imshow("GripSense", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("✅ Session ended!")
    




if __name__ == "__main__":
    main()

✅ YOLO loaded!
✅ MediaPipe loaded!
GripSense Running... Press Q to quit
✅ Session ended!
